## Introduction to the extended version of DiCE (Diverse Counterfactual Explanations)

[Mothilal et al. (2020)](https://dl.acm.org/doi/10.1145/3351095.3372850) introduce their method of generating counterfactual explanations considering _feasibility_, and _diversity_. [Guidotti and Ruggieri (2021)](https://link.springer.com/chapter/10.1007/978-3-030-88942-5_28), claim counterfactual explanations to be robust they should be similar for similar instances when they explain. In this study, in a search to improve the quality and reliability of the counterfactual explanations _robustness_ is found to be helpful and it also introduced in the optimization function.

DiCE-Extended is built upon the [DiCE (Diverse Counterfactual Explanations)](https://github.com/interpretml/DiCE) [(Mothilal et al. 2020)](https://dl.acm.org/doi/10.1145/3351095.3372850) framework by introducing a robustness term in the optimization function.

## Manipulated Optimization Function

The core enhancement in DiCE-Extended is the manipulated optimization function, designed to balance proximity, diversity, and feasibility of counterfactuals. The function is formulated as:

<a id="equation-1"></a>
\begin{equation}
\tag{1}
C(x) = \underset{c_1, ..., c_k}{\text{arg min}}
\frac{1}{2} \sum_{i=1}^{k} yloss(f(c_i), y) +
\frac{\lambda_1}{k} \sum_{i=1}^{k} dist(c_i, x) -
\lambda_2 \cdot dpp\_diversity(c_1, ..., c_k) -
\frac{\lambda_3}{k} \sum_{i=1}^{k} robustness(c_i, c_i')
\end{equation}

- **Proximity Loss**: The first term that averages the distance between generated counterfactuals and the original input ensure the counterfactuals to be as close as possible to the original input.
- **Diversity Loss**: Diversity of the counterfactual explanations is aquired by determinental point process of which loss is represented by the second term and it ensures that _k_ number of counterfactual explanations are generated.
- **Robustness Loss**: [Guidotti (2024)](https://link.springer.com/article/10.1007/s10618-022-00831-6) defines robustness as necessity of similar instances being explained by similar counterfactual explanations such that if $b(x_1)=b(x_2)=y$ then an explainer $f$ should generate counterfactuals $c_1$ and $c_2$ that are similar and can explain $x_1$ and $x_2$. The robustness term that is based on [Dice-Sørensen Coefficient](https://en.wikipedia.org/wiki/Dice-S%C3%B8rensen_coefficient), is adopted from [Bonasera and Carrizosa (2024)](
https://doi.org/10.48550/arXiv.2407.00843).

\begin{equation}
\tag{2}
Robustness(c_i, c_i') = \frac{2 * \lvert c_i \cap c_i' \rvert}{\lvert c_i \rvert + \lvert c_i' \rvert}
\end{equation}


By adjusting the weights $\lambda_1$, $\lambda_2$, $\lambda_3$ counterfactual explanations can be customised by specific needs.

## Metrics and Sensitivity Analysis for Dice Extended


### 1. Robustness Metrics

#### Dice-Sørensen Coefficient

To evaluate robustness, the Dice-Sørensen coefficient measures the similarity between counterfactuals c1 and
c2 generated for similar input instances x1 and x2:

\begin{equation}
\tag{3}
Robustness(c_1, c_2) = \frac{2 * \lvert c_1 \cap c_2 \rvert}{\lvert c_1 \rvert + \lvert c_2 \rvert}
\end{equation}

where:
- $ c_1 $ and $ c_2 $ are binary vectors,
- $ \lvert c_1 \cap c_2 \rvert $: The number of shared (overlapping) features between c1 and c2,
- $ \lvert c_1 \rvert $ and $ \lvert c_2 \rvert $: The total number of features in each counterfactual.

#### Input Perturbation and Stability

Stability under input perturbation measures the solution variance when slight perturbations are introduced
to the input instance. The procedure includes the following steps:

1) **Apply Gaussian Noise:** Perturb the input $x$ by adding Gaussian noise $\delta$ to create perturbed inputs
$x'$:

\begin{equation}
\tag{4}
x' = x + \delta, \quad \delta \sim \mathcal{N}(0, \sigma^2)
\end{equation}

where $\sigma$ is the standard deviation of the noise (e.g., $\sigma = 0.01$).

2) **Generate Counterfactuals:** Generate counterfactual explanations $c_i$ for the original input $x$ and $c_i'$ for the perturbed input $x'$.

3) **Measure Stability:** Compare counterfactuals using a distance metric, such as the Euclidean distance:

\begin{equation}
\tag{5}
Stability = \frac{1}{n} \sum_{i=1}^{n} dist(c_i, c_i')
\end{equation}

where:

\begin{equation}
\tag{6}
dist(c_i, c_i') = \sqrt{\sum_{j=1}^{d} (c_{ij} - c_{ij}')^2}
\end{equation}

$n$ is the total number of input instances, $c_i$ is the counterfactual for the original input, and $c_i'$ is the counterfactual for the perturbed input.

### 2. Counterfactual Quality Measures

#### Fidelity

Fidelity measures how often generated counterfactuals successfully change the model’s prediction:

\begin{equation}
\tag{7}
Fidelity = \frac{\sum_{i=1}^{n} \mathbf{1}(f(c_i) = y_{desired})}{n}
\end{equation}

where:

- $f$: Prediction model,
- $c_i$: Counterfactual instance,
- $y_{desired}$: Target output class,
- $n$: Total number of counterfactuals.

#### Proximity

Proximity measures the average distance between counterfactuals $c_i$ and the original inputs $x_i$:

\begin{equation}
\tag{8}
Proximity = \frac{1}{n} \sum_{i=1}^{n} dist(x_i, c_i)
\end{equation}

The Manhattan distance can be used for simplicity:

\begin{equation}
\tag{9}
dist(x_i, c_i) = \sum_{j=1}^{d} \lvert x_{ij} - c_{ij} \rvert
\end{equation}

#### Diversity

Diversity measures how dissimilar the counterfactuals $c_1, c_2, c_3,\ldots,c_k$ are among themselves:

\begin{equation}
\tag{10}
Diversity = \frac{1}{k(k-1)}\sum_{i_1}^{k}\sum_{j \neq i}^{} dist(c_i, c_j)
\end{equation}

where $k$ is the number of counterfactuals.

### 3. Sensitivity Analysis

#### Objective Function with Weights

The modified loss function in DiCE-Extended is defined as in the [equation 1](#equation-1) where:

- $yloss(f(c_i), y)$: Prediction loss for counterfactual instance $c_i$ relative to the desired outcome $y$,
- $dist(c_i, x)$: Distance metric (e.g., Euclidean or Manhattan) between the counterfactual c_i and the original input $x$,
- $dpp\_diversity(c_1,\ldots,c_k)$: Diversity loss term based on Determinantal Point Process (DPP),
- $Robustness(c_i,c_i')$: Robustness loss measuring similarity of counterfactuals under perturbations.

  The weights $\lambda_1, \lambda_2, \lambda_3$ control the balance between proximity, diversity, and robustness, respectively.

#### Sensitivity Analysis

To perform sensitivity analysis:

1) Vary the weights $\lambda_1, \lambda_2, \lambda_3$ systematically while ensuring:

\begin{equation}
\tag{11}
\lambda_1 + \lambda_2 + \lambda_3 = 1 (for normalization).
\end{equation}

2) Track the changes in the following metrics:

\begin{equation}
\tag{12}
P(\lambda_1, \lambda_2, \lambda_3) = Proximity,
\end{equation}

\begin{equation}
\tag{13}
D(\lambda_1, \lambda_2, \lambda_3) = Diversity,
\end{equation}

\begin{equation}
\tag{14}
R(\lambda_1, \lambda_2, \lambda_3) = Robustness,
\end{equation}

3) Measure the relationship between these metrics and the weights.



In [33]:
import sys
dice_path = "/Users/volk/Documents/bau24-25/thesis/repos/DiCE-X"
sys.path.insert(0, dice_path)

In [34]:
from dice_ml_x.utils import helpers

In [35]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [120]:
from dice_ml_x.benchmarking import Benchmarking
datasets = ["compas-recidivism", "adult-income", "lending-club", "german-credit"]
backends = ["sklearn", "PYT", "TF2"]
methods = ['gaussian', 'random', 'spherical']
benchmarking = Benchmarking(datasets=datasets,
                            backends=backends,
                            perturbation_methods=methods)
benchmarking.load_and_train(batch_size=16)

""" At the end of the load and train process a dictionary containing the training and counterfactual
generation history is saved following object can be found in the results object:
"""

Benchmarking:   0%|          | 0/12 [00:00<?, ?it/s]

the dataset is : compas-recidivism, the backend is : sklearn, the method is gaussian


100%|██████████| 1/1 [00:07<00:00,  7.87s/it]


the dataset is : compas-recidivism, the backend is : sklearn, the method is random


100%|██████████| 1/1 [00:39<00:00, 39.95s/it]


...

Diverse Counterfactuals found! total time taken: 04 min 42 sec
the dataset is : german-credit, the backend is : TF2, the method is random


Diverse Counterfactuals found! total time taken: 06 min 31 sec
the dataset is : german-credit, the backend is : TF2, the method is spherical


Benchmarking: 100%|██████████| 12/12 [6:12:56<00:00, 1864.69s/it, dataset=german-credit, backend=TF2, method=spherical]

Diverse Counterfactuals found! total time taken: 17 min 21 sec


' At the end of the load and train process a dictionary containing the training and counterfactual\ngeneration history is saved following object can be found in the results object:\n'

In [123]:
print(f"Model accuracy is {benchmarking.results['compas-recidivism']['PYT']['accuracy']} on the test dataset.")

Model accuracy is 0.6629778672032193 on the test dataset.


In [86]:
from IPython.display import display

original_instance = benchmarking.results['german-credit']['PYT']['input_instance']['gaussian']
gaussian_cfs = benchmarking.results['german-credit']['PYT']['cfs']['gaussian']

display(original_instance)
display(gaussian_cfs)

,status_of_existing_checking_account,duration_in_month,credit_history,purpose,credit_amount,savings_account_bonds,present_employment_since,installment_rate_in_percentage_of_disposable_income,personal_status_and_sex,other_debtors_guarantors,present_residence_since,property,age_in_years,other_installment_plans,housing,number_of_existing_credits_at_this_bank,job,number_of_people_being_liable_to_provide_maintenance_for,telephone,foreign_worker
128,0 <= ... < 200 DM,12,critical account/ other credits existing (not ...,car (used),1860,A61,unemployed,4,male : single,none,2,"if not A121/A122 : car or other, not in attrib...",34,none,own,2,management/ self-employed/ highly qualified em...,1,"yes, registered under the customers name",yes


,status_of_existing_checking_account,duration_in_month,credit_history,purpose,credit_amount,savings_account_bonds,present_employment_since,installment_rate_in_percentage_of_disposable_income,personal_status_and_sex,other_debtors_guarantors,...,property,age_in_years,other_installment_plans,housing,number_of_existing_credits_at_this_bank,job,number_of_people_being_liable_to_provide_maintenance_for,telephone,foreign_worker,credit_risk
0,0 <= ... < 200 DM,12,all credits at this bank paid back duly,business,2071,A61,unemployed,4,male : single,guarantor,...,"if not A121/A122 : car or other, not in attrib...",24,none,own,2,unemployed/ unskilled - non-resident,1,"yes, registered under the customers name",yes,0.765
1,no checking account,12,delay in paying off in the past,car (used),1828,A61,unemployed,4,male : single,co-applicant,...,"if not A121/A122 : car or other, not in attrib...",34,none,own,2,unemployed/ unskilled - non-resident,1,"yes, registered under the customers name",no,0.701
2,0 <= ... < 200 DM,12,delay in paying off in the past,car (used),1733,A63,... < 1 year,4,male : single,none,...,"if not A121/A122 : car or other, not in attrib...",34,none,own,2,unemployed/ unskilled - non-resident,1,none,yes,0.725
3,0 <= ... < 200 DM,18,all credits at this bank paid back duly,car (used),1786,A61,unemployed,4,male : single,guarantor,...,"if not A121/A122 : car or other, not in attrib...",34,none,for free,4,management/ self-employed/ highly qualified em...,1,none,yes,0.744


In [121]:
import pickle

with open("benchmarking_results_ran_w_2.pkl", "wb") as f:
    pickle.dump(benchmarking.results, f)

In [122]:
with open("benchmarking_results_ran_w_2.pkl", "rb") as f:
    b_results = pickle.load(f)

b_results

{'compas-recidivism': {'sklearn': {'accuracy': 0.5714285714285714,
   'cfs': {'gaussian':       sex   age  priors_count              race c_charge_degree  twoyearrecid
    0    Male  27.0           0.0  African-American               F             1
    0    Male  28.0           0.0  African-American               F             1
    0    Male  26.0           0.0  African-American               F             1
    0  Female  26.0           0.0         Caucasian               F             1,
    'random':       sex   age  priors_count              race c_charge_degree  twoyearrecid
    0    Male  27.0           0.0  African-American               F             1
    0  Female  18.0           0.0         Caucasian               F             1
    0    Male  18.0           0.0  African-American               F             1
    0    Male  18.0           0.0  African-American               M             1,
    'spherical':       sex   age  priors_count              race c_charge_degree  